In [1]:
import pandas  as  pd
from os import listdir

# TOMA DE  DATOS

In [2]:
def getData(path):
    dpath=[x for x in listdir(path) if '.' not in x]
    dpath=path+"/"+dpath[0]
    
    filesJ=[x for x in listdir(dpath) if '.json'in x]
    filesC=[x for x in listdir(dpath) if '.csv' in x]
    fileX=[x for x in listdir(dpath) if '.xml' in x]
    
    files={'json':filesJ,'csv':filesC,'xlm':fileX}
    result=pd.DataFrame()
    for t,l in files.items():
        
        if t=="json" and len(l)>0:
            data=[]
            for i  in l:
                try:
                    
                    d=pd.read_json(dpath+"/"+i)
                except ValueError:
                    d=pd.read_json(dpath+"/"+i, lines=True)
                data.append(d)
        elif t=="xlm" and len(l)>0:
            data=[pd.read_xml(dpath+"/"+x,xpath=".//Problem") for  x  in l]
        else:
            data=[]
        if  len(data)>0:
            result=pd.concat([result,pd.concat(data)])
    return result   
    
    

In [3]:
root=r"C:\Users\Gabo\Desktop\PAPPER Rlaif\createDataset\rawDataSet"
ps=listdir(root)
dataSets={}
for p in ps :
    print(p)
    b=getData(root+"/"+p)
    dataSets[p]=b
    print(b.columns, b.shape)
    

grade-school-math
Index(['question', 'ground_truth', '6b_finetuning', '6b_verification',
       '175b_finetuning', '175b_verification', 'answer'],
      dtype='object') (18903, 7)
MathDataset-ElementarySchool
Index(['category', 'subcategory', 'question', 'answer', 'reasoning', 'source'], dtype='object') (4693, 6)
mathQA
Index(['Problem', 'Rationale', 'options', 'correct', 'annotated_formula',
       'linear_formula', 'category'],
      dtype='object') (37901, 7)
nlu-asdiv-dataset
Index(['ID', 'Grade', 'Source', 'Body', 'Question', 'Solution-Type', 'Answer',
       'Formula', 'Class'],
      dtype='object') (2305, 9)
SVAMP
Index(['ID', 'Body', 'Question', 'Equation', 'Answer', 'Type'], dtype='object') (1000, 6)


In [4]:
info=[]
for i,j  in  dataSets.items():
    print(i,j.shape[0])
    a=list(zip([i for x in range(len(j))],j))
    info.extend(a)
info=pd.DataFrame(columns=['dataset','column'], data=info)

grade-school-math 18903
MathDataset-ElementarySchool 4693
mathQA 37901
nlu-asdiv-dataset 2305
SVAMP 1000


In [5]:
dataSets['MathDataset-ElementarySchool']['category'].value_counts()

category
Word Problems    1995
Geometry         1698
Arithmetic       1000
Name: count, dtype: int64

In [6]:
dataSets['SVAMP']

,ID,Body,Question,Equation,Answer,Type
0,chal-1,Each pack of dvds costs 76 dollars. If there i...,How much do you have to pay to buy each pack?,( 76.0 - 25.0 ),51,Subtraction
1,chal-2,Dan had $ 3 left with him after he bought a ca...,How much did the candy bar cost?,( 4.0 - 3.0 ),1,Subtraction
2,chal-3,Paco had 26 salty cookies and 17 sweet cookies...,How many salty cookies did Paco have left?,( 26.0 - 9.0 ),17,Subtraction
3,chal-4,43 children were riding on the bus. At the bus...,How many children got off the bus at the bus s...,( 43.0 - 21.0 ),22,Subtraction
4,chal-5,28 children were riding on the bus. At the bus...,How many more children got on the bus than tho...,( 30.0 - 28.0 ),2,Subtraction
...,...,...,...,...,...,...
995,chal-996,Paige was helping her mom plant flowers and to...,How many flower beds did they have?,( 36.0 / 12.0 ),3,Common-Division
996,chal-997,"At the zoo, a cage had 3 snakes and 75 alligat...",How many alligators were not hiding?,( 75.0 - 19.0 ),56,Subtraction
997,chal-998,Paige was helping her mom plant flowers and to...,How many flowers did they grow?,( 60.0 * ( 55.0 / 15.0 ) ),220,Multiplication
998,chal-999,Mary is baking a cake. The recipe calls for 7 ...,How many more cups of sugar does she need to add?,( 7.0 - 4.0 ),3,Subtraction


## Creación de  datasets y standar

In [7]:
dataSets['mathQA']['pasos']=dataSets['mathQA']['linear_formula'].apply(
    lambda x: len(x.split('|'))
)

standar=dataSets['nlu-asdiv-dataset']
print(standar.columns)

elementary=dataSets['MathDataset-ElementarySchool']
print(elementary.columns)

matqa=dataSets['mathQA']
print(matqa.columns)

svmap=dataSets['SVAMP']



Index(['ID', 'Grade', 'Source', 'Body', 'Question', 'Solution-Type', 'Answer',
       'Formula', 'Class'],
      dtype='object')
Index(['category', 'subcategory', 'question', 'answer', 'reasoning', 'source'], dtype='object')
Index(['Problem', 'Rationale', 'options', 'correct', 'annotated_formula',
       'linear_formula', 'category', 'pasos'],
      dtype='object')


## Conversion Standar (nlu-asdiv-dataset)

In [8]:
standar['pregunta']=standar[['Body','Question']].apply(
    lambda x: x['Body']+"\n"+x['Question'], axis=1)

replace={
    'Sum':'Addition',
    'Common-Division':'Division',
    'Floor-Division':'Division', 'Ceil-Division':'Division',
    'Algebra-1':'Algebra',
    'Algebra-2':'Algebra',
    'TVQ-Final':"TVQ",
    "TVQ-Change":"TVQ",
    'TVQ-Initial':"TVQ"
}

In [9]:
standar1=standar.replace(replace)
b=standar1['Solution-Type'].value_counts()
c=b[b<40]
print(c.index, sum(c))
standar2=standar1.replace(dict(zip(c.index,[pd.NA for x in c.index])))
standar2.dropna(subset='Solution-Type',inplace=True)

Index(['Sequential-Operation', 'Set-Operation', 'UnitTrans',
       'Number-Operation', 'Number-Pattern', 'Substraction'],
      dtype='object', name='Solution-Type') 60


# Transformación de  datos
Transfromamos los  datos  al  standar

## Conversion mathQA

In [10]:
print(matqa.columns)

Index(['Problem', 'Rationale', 'options', 'correct', 'annotated_formula',
       'linear_formula', 'category', 'pasos'],
      dtype='object')


In [11]:
a=matqa['linear_formula'].values[138]
print(a)

multiply(const_10,const_4)|subtract(n2,n0)|subtract(n1,n3)|divide(#1,#2)|multiply(n1,#3)|add(n0,#4)|subtract(#5,#0)


In [12]:
categories=list(map(
    lambda x:  map( lambda y: y[: y.find('(')],x.split('|') ) ,
    matqa['linear_formula'].values
))
from itertools import chain
categories=set(chain.from_iterable(categories))
categories=[x for x in categories if len(x)>=2]
print(categories)

['add', 'square_edge_by_area', 'speed', 'floor', 'volume_cylinder', 'log', 'triangle_area', 'cube_edge_by_volume', 'inverse', 'tangent', 'negate_prob', 'volume_cube', 'max', 'multiply', 'circle_area', 'speed_in_still_water', 'quadrilateral_area', 'cosine', 'choose', 'volume_rectangular_prism', 'circumface', 'gcd', 'volume_sphere', 'negate', 'lcm', 'min', 'square_edge_by_perimeter', 'square_area', 'sine', 'rectangle_perimeter', 'triangle_perimeter', 'p_after_gain', 'surface_rectangular_prism', 'rhombus_perimeter', 'diagonal', 'subtract', 'power', 'original_price_before_loss', 'original_price_before_gain', 'factorial', 'surface_cylinder', 'square_perimeter', 'permutation', 'stream_speed', 'rhombus_area', 'surface_cube', 'surface_sphere', 'divide', 'reminder', 'rectangle_area', 'triangle_area_three_edges', 'sqrt', 'volume_cone']


In [13]:
def  getOperations(x:str,catego:list)->tuple:
    values=tuple(map(lambda cat: x.count(cat),catego ))
    return values

data=list(map(lambda x: getOperations(x,categories), matqa['linear_formula'].values))
operations=pd.DataFrame(data=data, columns=categories)

for i in ['Problem','linear_formula', 'category', 'pasos']:
    operations[i]=matqa[i].values

operations['pasos']=operations.iloc[::,:-4].sum(axis=1)

In [14]:
standar2['Solution-Type'].value_counts()

Solution-Type
Subtraction       452
Addition          446
Division          308
Multiplication    260
Algebra           154
TVQ               119
Ratio              94
Geometry           88
GCD                73
Surplus            70
LCM                65
Difference         59
Comparison         57
Name: count, dtype: int64

# Creación base  de  entrenamiento de cabezales

In [15]:
def level(x:int):
    if x<=5:
        return "ELEMENTARY"
    elif x>5 and x<=8:
        return "MIDDLE"
    else:
        return "HIGH"

In [16]:
standar2.drop(columns='Class', inplace=True)

In [17]:
standar2['Grade'].value_counts()

Grade
3    805
6    500
2    335
4    301
1    195
5    109
Name: count, dtype: int64

In [18]:
base=standar2[['pregunta','Grade','Solution-Type']]
base['level']='elementary'
base['claridad']=1
base['signature']='mathematics'


C:\Users\Gabo\AppData\Local\Temp\ipykernel_39788\1018747540.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['level']='elementary'
C:\Users\Gabo\AppData\Local\Temp\ipykernel_39788\1018747540.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['claridad']=1
C:\Users\Gabo\AppData\Local\Temp\ipykernel_39788\1018747540.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docu

In [19]:
base['level']=base['Grade'].apply(level)

C:\Users\Gabo\AppData\Local\Temp\ipykernel_39788\1447880555.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['level']=base['Grade'].apply(level)


In [20]:
base['level'].value_counts()

level
ELEMENTARY    1745
MIDDLE         500
Name: count, dtype: int64

In [21]:
# Definimos la  habilidad.
def getHability(x, columns):
    a=x.iloc[:-4]
    ind=a.argmax()
    return columns[ind]

operations['hability']=operations.apply(lambda x: getHability(x,operations.columns[:-4]),axis=1)

In [22]:
operations['category'].value_counts()

category
general        16861
physics         8962
gain            6478
geometry        2691
other           2314
probability      595
Name: count, dtype: int64

In [23]:
primary=['general', 'gain', 'geometry', 'other']
matqa1=operations.copy()
matqa1['nivel']=matqa1[['pasos','category']].apply(lambda x: 'primaria'
                                                   if x['pasos']<= 3 and x['category'] in primary
                                                   else 'otro'
                                                   , axis=1)

nolevel=matqa1[(matqa1['nivel']=='otro')&(~matqa1['category'].isin(primary))]

nolevel['claridad']=1
nolevel['hability']='otro'

C:\Users\Gabo\AppData\Local\Temp\ipykernel_39788\610032371.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nolevel['claridad']=1
C:\Users\Gabo\AppData\Local\Temp\ipykernel_39788\610032371.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nolevel['hability']='otro'


In [24]:
programing=pd.read_csv(r"C:\Users\Gabo\Desktop\PAPPER Rlaif\createDataset\nosignature\programing.csv")
circuits=pd.read_csv(r"C:\Users\Gabo\Desktop\PAPPER Rlaif\createDataset\nosignature\circuits.csv")

In [25]:
programing=programing[['Category folder (optional)','Question title',
       'Question text', 'Question Type']]
circuits=circuits[['Category folder (optional)','Question title',
       'Question text', 'Question Type']]
def  haveimg(x):
       return not '<img' in x
programing['img']=programing['Question text'].apply(haveimg)
circuits['img']=circuits['Question text'].apply(haveimg)
programing=programing[programing['img']]
circuits=circuits[circuits['img']]
print(programing.shape)
print(circuits.shape)
circuits['level']='OTHER'
circuits['signature']='electronic'

(0, 5)
(205, 5)


In [57]:
scienceEasygrade=pd.read_csv(r"C:\Users\Gabo\Desktop\PAPPER Rlaif\createDataset\nosignature\ARC-Easy-Test.csv")
scienceEasygrade['level']=scienceEasygrade['schoolGrade'].apply(level)
scienceEasygrade['signature']='science'

In [27]:
scienceEasygrade['level'].value_counts()

level
MIDDLE        1325
ELEMENTARY     859
HIGH           192
Name: count, dtype: int64

# Conglomerado de  bases

In [28]:
from sklearn.model_selection import train_test_split

In [29]:
base.rename(columns={'Solution-Type':'skill'}, inplace=True)
base.drop(columns="Grade", inplace=True)

C:\Users\Gabo\AppData\Local\Temp\ipykernel_39788\336207233.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base.rename(columns={'Solution-Type':'skill'}, inplace=True)
C:\Users\Gabo\AppData\Local\Temp\ipykernel_39788\336207233.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base.drop(columns="Grade", inplace=True)


In [30]:
base.columns

Index(['pregunta', 'skill', 'level', 'claridad', 'signature'], dtype='object')

In [31]:
# No  nivel = nolevel
# No signature= science, circuits
# Hability
# Clarity


In [50]:
print(nolevel.columns)
nolevel.rename(
    columns={'Problem':"pregunta",
             "Pregunta":'pregunta',
             'hability':"skill",
             'category':"signature",
             "nivel":'level'}
, inplace=True)
nolevel1=nolevel[base.columns].copy()

Index(['add', 'square_edge_by_area', 'speed', 'floor', 'volume_cylinder',
       'log', 'triangle_area', 'cube_edge_by_volume', 'inverse', 'tangent',
       'negate_prob', 'volume_cube', 'max', 'multiply', 'circle_area',
       'speed_in_still_water', 'quadrilateral_area', 'cosine', 'choose',
       'volume_rectangular_prism', 'circumface', 'gcd', 'volume_sphere',
       'negate', 'lcm', 'min', 'square_edge_by_perimeter', 'square_area',
       'sine', 'rectangle_perimeter', 'triangle_perimeter', 'p_after_gain',
       'surface_rectangular_prism', 'rhombus_perimeter', 'diagonal',
       'subtract', 'power', 'original_price_before_loss',
       'original_price_before_gain', 'factorial', 'surface_cylinder',
       'square_perimeter', 'permutation', 'stream_speed', 'rhombus_area',
       'surface_cube', 'surface_sphere', 'divide', 'reminder',
       'rectangle_area', 'triangle_area_three_edges', 'sqrt', 'volume_cone',
       'pregunta', 'linear_formula', 'signature', 'pasos', 'skill', 'lev

C:\Users\Gabo\AppData\Local\Temp\ipykernel_39788\748024742.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nolevel.rename(


In [58]:
scienceEasygrade.rename(columns={
    'question':'pregunta'
}, inplace=True)
scienceEasygrade['claridad']=1
scienceEasygrade['skill']='otro'
science=scienceEasygrade[base.columns].copy()

In [34]:
elementary.columns

Index(['category', 'subcategory', 'question', 'answer', 'reasoning', 'source'], dtype='object')

In [35]:
base.columns

Index(['pregunta', 'skill', 'level', 'claridad', 'signature'], dtype='object')

In [36]:
elementary['level']='ELEMENTARY'
elementary['claridad']=1
elementary['signature']='mathematics'
elementary.rename(columns={
    'subcategory':'skill',
    'question':"pregunta"

}, inplace=True)

elementary1=elementary[base.columns].copy()

In [37]:
svmap['level']='elementary'
svmap['pregunta']=svmap[['Body','Question']].apply(
    lambda x: x['Body']+"\n"+x['Question'], axis=1)
svmap.rename(columns={
    'Type':'skill',
    
}, inplace=True)
svmap['claridad']=1
svmap['signature']='mathematics'
print(svmap['pregunta'].values[0])
svmap1=svmap[base.columns].copy()

Each pack of dvds costs 76 dollars. If there is a discount of 25 dollars on each pack
How much do you have to pay to buy each pack?


In [38]:
#unimos  las  bases  positivas
base=pd.concat([base,svmap1,elementary1])

In [59]:
baseN=pd.concat([science,nolevel1])

In [60]:
basef=pd.concat([base,baseN])

In [61]:
basef['skill'].value_counts()

skill
otro                  11933
geometry               1698
challenge              1000
Subtraction             983
Addition                641
multi_step              600
add_sub                 395
Multiplication          368
Division                308
Common-Division         165
Algebra                 154
TVQ                     119
Ratio                    94
Geometry                 88
div_remainder            73
GCD                      73
sequence_next_term       72
gcd                      72
lcm                      72
mul                      72
div                      71
conversion               71
add_sub_multiple         71
arithmetic_mixed         71
add_or_sub               71
mul_div_multiple         71
round_number             71
time                     71
place_value              71
Surplus                  70
LCM                      65
Difference               59
Comparison               57
Common-Divison            1
Name: count, dtype: int64

In [62]:
basef['skill'].unique()

array(['Addition', 'Subtraction', 'Comparison', 'TVQ', 'Multiplication',
       'Division', 'Surplus', 'Geometry', 'Difference', 'Ratio', 'LCM',
       'GCD', 'Algebra', 'Common-Division', 'Common-Divison',
       'arithmetic_mixed', 'sequence_next_term', 'add_or_sub',
       'add_sub_multiple', 'div', 'mul', 'mul_div_multiple', 'conversion',
       'time', 'div_remainder', 'gcd', 'lcm', 'place_value',
       'round_number', 'geometry', 'add_sub', 'multi_step', 'challenge',
       'otro'], dtype=object)

In [83]:
replaceSkill={'addition':'addition', 
              'subtraction':"subtraction",
              "tvq":'time',
              'geometry':'geometry',
              'common-division':'division',
              'common-divison':'division',
              'arithmetic_mixed':'algebra',
              'add_sub':'addition-subtraction',
              'add_or_sub':'addition-subtraction',
              'div':'division',
              'mul':'multiplication',
              'div_remainder':'division',
              'difference':'comparison',
              'add_sub_multiple':'addition-subtraction',
              'mul_div_multiple':'algebra',
              'multi_step':'algebra'
              }
basef['skill']=basef['skill'].str.lower()
basef['skill'].replace(replaceSkill, inplace=True)
a=basef['skill'].value_counts()
a=a[a<90]
basef1=basef.copy()
basef1=basef1[~basef1['skill'].isin(a.index)]
print(basef1['skill'].value_counts())

skill
otro                    11933
geometry                 1786
challenge                1000
subtraction               983
algebra                   896
addition                  641
division                  618
addition-subtraction      537
multiplication            440
time                      190
gcd                       145
lcm                       137
comparison                116
ratio                      94
Name: count, dtype: int64


C:\Users\Gabo\AppData\Local\Temp\ipykernel_39788\2916970344.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  basef['skill'].replace(replaceSkill, inplace=True)


In [85]:
basef1['signature'].value_counts()

signature
physics        8962
mathematics    7583
science        2376
probability     595
Name: count, dtype: int64

In [90]:
basef1['level']=basef1['level'].str.upper()
basef1['level'].value_counts()

level
OTRO          9557
ELEMENTARY    7942
MIDDLE        1825
HIGH           192
Name: count, dtype: int64

In [ ]:
#base de  datos  con preguntas inconsistentes
noclarity=pd.read_csv(r"C:\Users\Gabo\Desktop\PAPPER Rlaif\createDataset\samplequestions_baja_claridad.csv", index_col=0)

In [101]:
noclarity['claridad']=0

In [103]:
baseff=pd.concat([basef1,noclarity])

In [112]:
baseff.rename(columns={
    'signature':'subject'
}, inplace =True)

In [113]:
baseff.to_csv("baseCabezales.csv")